# 2) Interactive Adapter Training (Colab)

Widget-driven continual adapter training with live progress and OOD calibration.

In [ ]:
import json
import os
import random
import shutil
import subprocess
import sys
import time
from pathlib import Path
from datetime import datetime

import matplotlib.pyplot as plt
import torch

def _ensure_repo_on_path() -> Path | None:
    markers = ("scripts", "src", "colab_notebooks")
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path('/content/bitirme projesi'),
        Path('/content/bitirmeprojesi'),
        Path('/content/drive/MyDrive/bitirme projesi'),
        Path('/content/drive/MyDrive/bitirmeprojesi'),
    ]
    seen = set()
    for base in candidates:
        try:
            probes = [base, *list(base.parents)[:3]]
            for probe in probes:
                key = str(probe)
                if key in seen:
                    continue
                seen.add(key)
                if all((probe / marker).exists() for marker in markers):
                    os.chdir(probe)
                    if str(probe) not in sys.path:
                        sys.path.insert(0, str(probe))
                    return probe
        except Exception:
            continue
    return None


def _auto_clone_repo_for_colab() -> Path | None:
    repo_url = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')
    target = Path('/content/bitirmeprojesi')
    try:
        if target.exists() and (target / '.git').exists():
            subprocess.run(['git', '-C', str(target), 'pull', 'origin', 'master'], check=False)
            return target
        if target.exists() and not (target / '.git').exists():
            shutil.rmtree(target, ignore_errors=True)
        subprocess.run(['git', 'clone', '--depth', '1', repo_url, str(target)], check=True)
        return target
    except Exception as clone_exc:
        print(f'[bootstrap] Auto-clone failed: {clone_exc}')
        return None


_repo_root_hint = _ensure_repo_on_path()
if _repo_root_hint is None and ('COLAB_RELEASE_TAG' in os.environ or 'google.colab' in sys.modules):
    cloned = _auto_clone_repo_for_colab()
    if cloned is not None:
        _repo_root_hint = _ensure_repo_on_path()


try:
    from scripts.colab_repo_bootstrap import (
        install_colab_requirements,
        mount_drive_if_available,
        resolve_repo_root,
        running_in_colab,
    )
except ModuleNotFoundError as exc:
    searched = [
        str(Path.cwd()),
        str(Path('/content/bitirme projesi')),
        str(Path('/content/bitirmeprojesi')),
        str(Path('/content/drive/MyDrive/bitirme projesi')),
        str(Path('/content/drive/MyDrive/bitirmeprojesi')),
    ]
    raise RuntimeError(
        "Could not import scripts.colab_repo_bootstrap after auto-bootstrap. Ensure git access is available or set AADS_REPO_URL. "
        + f"Detected root: {_repo_root_hint}. Searched: {searched}"
    ) from exc



def rt(msg: str) -> None:
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")


IN_COLAB = running_in_colab()

rt("Cell 1: setup started")
if IN_COLAB:
    mount_drive_if_available(force_remount=False)

ROOT = resolve_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

REQ = ROOT / "colab_notebooks" / "requirements_colab.txt"
install_colab_requirements(REQ, IN_COLAB)

try:
    import transformers as _tf
    TRANSFORMERS_VERSION = str(getattr(_tf, "__version__", "unknown"))
except Exception:
    TRANSFORMERS_VERSION = "unknown"

if torch.cuda.is_available():
    DEVICE = "cuda"
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
else:
    DEVICE = "cpu"

rt(f"Repository root: {ROOT}")
rt(f"Device: {DEVICE}")
rt(f"transformers version: {TRANSFORMERS_VERSION}")

if DEVICE != "cuda":
    raise RuntimeError(
        "GPU runtime is required. This notebook refuses CPU execution to enforce training parity. "
        "In Colab: Runtime -> Change runtime type -> GPU (A100/T4/L4), then rerun Cell 1."
    )

rt("Cell 1: setup completed")

In [ ]:
import ipywidgets as widgets
from IPython.display import display

rt("Cell 2: building parameter panel")

BASE_CONFIG = json.loads((ROOT / "config" / "base.json").read_text(encoding="utf-8"))
TRAIN_CFG = BASE_CONFIG.get("training", {}).get("continual", {})
ADAPTER_CFG = TRAIN_CFG.get("adapter", {})
OOD_CFG = TRAIN_CFG.get("ood", {})

STATE = {
    "validated": False,
    "dataset_summary": None,
    "class_names": [],
    "runtime_dataset_root": None,
    "adapter": None,
    "loaders": None,
    "history": None,
    "calibration": None,
}

dataset_root_text = widgets.Text(value="data/class_root_dataset", description="Dataset:", layout=widgets.Layout(width="700px"))
crop_name_text = widgets.Text(value="tomato", description="Crop:")
epochs_slider = widgets.IntSlider(value=int(TRAIN_CFG.get("num_epochs", 1)), min=1, max=50, description="Epochs")
batch_size_slider = widgets.IntSlider(value=int(TRAIN_CFG.get("batch_size", 8)), min=1, max=256, description="Batch")
lr_slider = widgets.FloatLogSlider(value=float(TRAIN_CFG.get("learning_rate", 1e-4)), base=10, min=-6, max=-2, step=0.05, description="LR")
lora_r_slider = widgets.IntSlider(value=int(ADAPTER_CFG.get("lora_r", 16)), min=4, max=64, description="LoRA r")
lora_alpha_slider = widgets.IntSlider(value=int(ADAPTER_CFG.get("lora_alpha", 32)), min=8, max=128, description="LoRA α")
lora_dropout_slider = widgets.FloatSlider(value=float(ADAPTER_CFG.get("lora_dropout", 0.1)), min=0.0, max=0.5, step=0.01, description="Dropout")
ood_slider = widgets.FloatSlider(value=float(OOD_CFG.get("threshold_factor", 2.0)), min=1.0, max=5.0, step=0.1, description="OOD x")
num_workers_slider = widgets.IntSlider(value=4, min=0, max=16, description="Workers")
prefetch_factor_slider = widgets.IntSlider(value=2, min=2, max=8, description="Prefetch")
persistent_workers_check = widgets.Checkbox(value=True, description="Persistent workers")
pin_memory_check = widgets.Checkbox(value=True, description="Pin memory")
use_cache_check = widgets.Checkbox(value=False, description="Dataset cache")
validate_btn = widgets.Button(description="Validate & Start Training", button_style="success")
status_out = widgets.Output()
guide_html = widgets.HTML(value=(
    "<b>Parameter Guide (Suggested Values)</b><br>"
    "<ul style='margin-top:6px'>"
    "<li><b>Dataset</b>: class-root folder like <code>data/class_root_dataset</code> (required).</li>"
    "<li><b>Crop</b>: crop adapter name (suggested: <code>tomato</code>, <code>pepper</code>, <code>corn</code>).</li>"
    "<li><b>Epochs</b>: total training passes (start with <b>3-10</b>, increase if underfitting).</li>"
    "<li><b>Batch</b>: images per step (suggested <b>8</b> on T4/16GB, <b>16-64</b> on larger GPUs).</li>"
    "<li><b>Workers</b>: data-loader CPU workers (start at <b>4</b>, try <b>8-12</b> on high-RAM runtimes).</li>"
    "<li><b>Prefetch</b>: batches prefetched per worker (use <b>2</b> first; increase if pipeline stalls).</li>"
    "<li><b>Persistent workers</b>: keep workers alive between epochs for lower loader overhead.</li>"
    "<li><b>Pin memory</b>: faster CPU→GPU transfer, usually keep enabled on GPU runtimes.</li>"
    "<li><b>LR</b>: learning rate (safe default <b>1e-4</b>; try <b>5e-5</b> if unstable).</li>"
    "<li><b>LoRA r</b>: adapter rank (suggested <b>16</b>; try <b>8</b> for speed, <b>32</b> for capacity).</li>"
    "<li><b>LoRA α</b>: scaling strength (suggested <b>32</b>; often about 2x <code>r</code>).</li>"
    "<li><b>Dropout</b>: LoRA regularization (suggested <b>0.1</b>; use <b>0.05-0.2</b>).</li>"
    "<li><b>OOD x</b>: OOD threshold factor (default <b>2.0</b>; increase to reduce false OOD).</li>"
    "</ul>"
    "Click <i>Validate & Start Training</i>, then run Cell 3 for dataset checks."
))


def gather_widget_config():
    return {
        "dataset_root": dataset_root_text.value.strip(),
        "crop_name": crop_name_text.value.strip(),
        "num_epochs": int(epochs_slider.value),
        "batch_size": int(batch_size_slider.value),
        "num_workers": int(num_workers_slider.value),
        "prefetch_factor": int(prefetch_factor_slider.value),
        "persistent_workers": bool(persistent_workers_check.value),
        "pin_memory": bool(pin_memory_check.value),
        "use_cache": bool(use_cache_check.value),
        "learning_rate": float(lr_slider.value),
        "lora_r": int(lora_r_slider.value),
        "lora_alpha": int(lora_alpha_slider.value),
        "lora_dropout": float(lora_dropout_slider.value),
        "ood_threshold_factor": float(ood_slider.value),
    }


def _on_validate_clicked(_):
    cfg = gather_widget_config()
    STATE["widget_config"] = cfg
    with status_out:
        status_out.clear_output()
        print("Config captured. Run Cell 3 for full dataset validation details.")
        print(f"Timestamp: {datetime.now().strftime('%H:%M:%S')}")


validate_btn.on_click(_on_validate_clicked)

panel = widgets.VBox([
    guide_html,
    dataset_root_text,
    crop_name_text,
    widgets.HBox([epochs_slider, batch_size_slider, num_workers_slider]),
    widgets.HBox([prefetch_factor_slider, persistent_workers_check, pin_memory_check, use_cache_check]),
    widgets.HBox([lr_slider, lora_r_slider, lora_alpha_slider]),
    widgets.HBox([lora_dropout_slider, ood_slider]),
    validate_btn,
    status_out,
]

display(panel)
rt("Cell 2: parameter panel ready")

In [ ]:
from scripts.evaluate_dataset_layout import evaluate_layout

rt("Cell 3: dataset validation started")

cfg = STATE.get("widget_config") or {
    "dataset_root": dataset_root_text.value.strip(),
    "crop_name": crop_name_text.value.strip(),
}

raw_root = Path(cfg["dataset_root"]).expanduser()
if not raw_root.is_absolute():
    raw_root = (ROOT / raw_root).resolve()

summary = evaluate_layout(raw_root)
STATE["dataset_summary"] = summary

print("Dataset root:", raw_root)
print("Layout ok:", summary.get("ok"))
print("Summary:", summary.get("summary", {}))

if summary.get("errors"):
    print("Errors:")
    for item in summary["errors"]:
        print(" -", item)

if summary.get("warnings"):
    print("Warnings:")
    for item in summary["warnings"]:
        print(" -", item)

class_names = [entry.get("class_name") for entry in summary.get("classes", []) if entry.get("class_name")]
STATE["class_names"] = class_names
print("Discovered classes:", class_names)

STATE["validated"] = bool(summary.get("ok")) and len(class_names) > 0
rt(f"Cell 3: validation completed (ok={STATE['validated']})")

In [ ]:
from src.adapter.independent_crop_adapter import IndependentCropAdapter
from src.utils.data_loader import create_training_loaders

rt("Cell 4: engine initialization started")


def prepare_runtime_dataset_layout(
    class_root: Path,
    crop_name: str,
    seed: int = 42,
    force_rebuild: bool = False,
    allowed_class_names: list[str] | None = None,
) -> Path:
    runtime_root = ROOT / "data" / "runtime_notebook_datasets"
    crop_root = runtime_root / crop_name
    metadata_path = crop_root / "_split_metadata.json"

    class_dirs = sorted([p for p in class_root.iterdir() if p.is_dir()])
    allowed_set = set(allowed_class_names or [])
    source_key = {
        "layout_version": 3,
        "class_root": str(class_root.resolve()),
        "crop_name": crop_name,
        "seed": int(seed),
        "class_dirs": [p.name for p in class_dirs],
        "allowed_class_names": sorted(list(allowed_set)),
        "split_policy": "continual=80,val=10,test=10",
    }

    if (not force_rebuild) and crop_root.exists() and metadata_path.exists():
        try:
            cached = json.loads(metadata_path.read_text(encoding="utf-8"))
        except Exception:
            cached = {}
        if cached == source_key:
            rt("Reusing cached runtime dataset split layout")
            return runtime_root

    if crop_root.exists():
        shutil.rmtree(crop_root)
    crop_root.mkdir(parents=True, exist_ok=True)

    rng = random.Random(seed)
    split_names = ("continual", "val", "test")
    image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

    def materialize_file(src_path: Path, dst: Path) -> None:
        try:
            os.symlink(src_path, dst)
            return
        except OSError:
            pass

        try:
            os.link(src_path, dst)
            return
        except OSError:
            pass

        shutil.copy2(src_path, dst)

    used_target_class_names = set()
    for cls_dir in class_dirs:
        normalized_class_name = normalize_class_name(cls_dir.name)
        if not normalized_class_name:
            continue
        if allowed_set and normalized_class_name not in allowed_set:
            continue
        if normalized_class_name in used_target_class_names:
            rt(f"Skipping duplicate normalized class name: {cls_dir.name} -> {normalized_class_name}")
            continue

        rt(f"Preparing dataset split for class: {cls_dir.name}")
        image_files = [
            p for p in cls_dir.rglob("*")
            if p.is_file() and p.suffix.lower() in image_exts
        ]
        rng.shuffle(image_files)
        n = len(image_files)
        if n == 0:
            continue

        train_n = max(1, int(0.8 * n))
        val_n = max(1, int(0.1 * n)) if n >= 3 else 0
        if train_n + val_n >= n:
            val_n = max(0, n - train_n - 1)
        test_n = max(0, n - train_n - val_n)

        split_map = {
            "continual": image_files[:train_n],
            "val": image_files[train_n:train_n + val_n],
            "test": image_files[train_n + val_n:train_n + val_n + test_n],
        }

        for split in split_names:
            target = crop_root / split / normalized_class_name
            target.mkdir(parents=True, exist_ok=True)
            for src_path in split_map[split]:
                dst = target / src_path.name
                if dst.exists():
                    continue
                materialize_file(src_path, dst)
        used_target_class_names.add(normalized_class_name)

    metadata_path.write_text(json.dumps(source_key, indent=2), encoding="utf-8")
    return runtime_root


if not STATE.get("validated"):
    raise RuntimeError("Run Cell 3 and ensure dataset validation passes first.")

cfg = STATE["widget_config"]
crop_name = cfg["crop_name"].strip().lower()
class_root = Path(cfg["dataset_root"]).expanduser()
if not class_root.is_absolute():
    class_root = (ROOT / class_root).resolve()

def normalize_class_name(name: str) -> str:
    normalized = "".join(ch.lower() if ch.isalnum() else "_" for ch in str(name).strip())
    while "__" in normalized:
        normalized = normalized.replace("__", "_")
    return normalized.strip("_")

available_class_names = sorted({
    normalize_class_name(p.name)
    for p in class_root.iterdir()
    if p.is_dir() and normalize_class_name(p.name)
})
taxonomy_path = ROOT / "config" / "plant_taxonomy.json"
expected_for_crop = None
try:
    taxonomy = json.loads(taxonomy_path.read_text(encoding="utf-8"))
    disease_map = taxonomy.get("crop_specific_diseases", {})
    crop_diseases = disease_map.get(crop_name, []) if isinstance(disease_map, dict) else []
    if isinstance(crop_diseases, list) and crop_diseases:
        normalized_diseases = [normalize_class_name(name) for name in crop_diseases]
        expected_for_crop = sorted({"healthy", *[name for name in normalized_diseases if name]})
except Exception as exc:
    rt(f"Taxonomy lookup failed ({exc}); using available class folders.")

if expected_for_crop:
    aligned_class_names = [name for name in expected_for_crop if name in available_class_names]
else:
    aligned_class_names = available_class_names

if not aligned_class_names:
    raise RuntimeError(
        f"No usable classes found for crop '{crop_name}'. Available normalized classes: {available_class_names}"
    )

STATE["class_names"] = aligned_class_names
print("Training classes:", aligned_class_names)

runtime_data_root = prepare_runtime_dataset_layout(
    class_root=class_root,
    crop_name=crop_name,
    force_rebuild=os.environ.get("AADS_FORCE_REBUILD_SPLITS", "0") == "1",
    allowed_class_names=aligned_class_names,
)
STATE["runtime_dataset_root"] = runtime_data_root
rt("Runtime dataset layout prepared")

continual_cfg = json.loads(json.dumps(BASE_CONFIG.get("training", {}).get("continual", {})))
continual_cfg["backbone"]["model_name"] = continual_cfg.get("backbone", {}).get("model_name", "facebook/dinov3-vitl16-pretrain-lvd1689m")
continual_cfg["device"] = DEVICE
continual_cfg["num_epochs"] = int(cfg["num_epochs"])
continual_cfg["batch_size"] = int(cfg["batch_size"])
continual_cfg["learning_rate"] = float(cfg["learning_rate"])
continual_cfg["adapter"]["lora_r"] = int(cfg["lora_r"])
continual_cfg["adapter"]["lora_alpha"] = int(cfg["lora_alpha"])
continual_cfg["adapter"]["lora_dropout"] = float(cfg["lora_dropout"])
continual_cfg["ood"]["threshold_factor"] = float(cfg["ood_threshold_factor"])

adapter = IndependentCropAdapter(crop_name=crop_name, device=DEVICE)
rt("Initializing adapter engine (this may download/load backbone)")
adapter.initialize_engine(class_names=STATE["class_names"], config={"training": {"continual": continual_cfg}})
rt("Adapter engine initialized")

loaders = create_training_loaders(
    data_dir=str(runtime_data_root),
    crop=crop_name,
    batch_size=int(cfg["batch_size"]),
    num_workers=0,
    use_cache=False,
)
rt("Training/validation/test loaders created")

STATE["adapter"] = adapter
STATE["loaders"] = loaders
STATE["continual_config"] = continual_cfg

print("Engine initialized.")
print("Runtime dataset root:", runtime_data_root)
print("Classes:", len(STATE["class_names"]))
print("Resolved LoRA target modules:", len(adapter.target_modules_resolved))
print("Fusion layers:", continual_cfg.get("fusion", {}).get("layers"))
print("Fusion output dim:", continual_cfg.get("fusion", {}).get("output_dim"))

trainer = adapter._trainer
if trainer is not None:
    total_params = 0
    trainable_params = 0
    for p in trainer.adapter_model.parameters():
        total_params += p.numel()
        if p.requires_grad:
            trainable_params += p.numel()
    for p in trainer.classifier.parameters():
        total_params += p.numel()
        if p.requires_grad:
            trainable_params += p.numel()
    for p in trainer.fusion.parameters():
        total_params += p.numel()
        if p.requires_grad:
            trainable_params += p.numel()

    print(f"Total params (adapter+heads): {total_params:,}")
    print(f"Trainable params: {trainable_params:,}")

rt("Cell 4: engine initialization completed")

In [ ]:
rt("Cell 4.5: loader micro-benchmark started")

if STATE.get("runtime_dataset_root") is None:
    raise RuntimeError("Run Cell 4 first.")
if "widget_config" not in STATE:
    raise RuntimeError("Run Cell 2 first to capture widget configuration.")

cfg = dict(STATE["widget_config"] or {})
crop_name = str(cfg.get("crop_name", "")).strip().lower()
runtime_data_root = STATE["runtime_dataset_root"]
if not crop_name:
    raise RuntimeError("Missing crop_name in widget config.")

base_batch = int(cfg.get("batch_size", 32))
base_workers = int(cfg.get("num_workers", 0))
prefetch_factor = int(cfg.get("prefetch_factor", 2))
persistent_workers = bool(cfg.get("persistent_workers", True))
pin_memory = bool(cfg.get("pin_memory", True))
use_cache = bool(cfg.get("use_cache", False))

batch_candidates = sorted({max(8, base_batch // 2), base_batch, min(256, max(base_batch + 8, base_batch * 2))})
worker_candidates = sorted({max(0, base_workers // 2), base_workers, min(16, max(base_workers + 2, base_workers * 2))})
combos = [(b, w) for b in batch_candidates for w in worker_candidates]

# Keep total benchmark around ~60s.
budget_seconds = 60.0
per_combo_seconds = max(5.0, budget_seconds / max(1, len(combos)))

def _build_train_loader(batch_size_value: int, workers_value: int):
    kwargs = {}
    if workers_value > 0:
        kwargs["prefetch_factor"] = prefetch_factor
        kwargs["persistent_workers"] = persistent_workers
    loaders_local = create_training_loaders(
        data_dir=str(runtime_data_root),
        crop=crop_name,
        batch_size=batch_size_value,
        num_workers=workers_value,
        use_cache=use_cache,
        pin_memory=pin_memory,
        **kwargs,
    )
    return loaders_local["train"], loaders_local

results = []
for batch_size_value, workers_value in combos:
    train_loader_local, loaders_local = _build_train_loader(batch_size_value, workers_value)

    steps = 0
    samples = 0
    started = time.time()
    for batch in train_loader_local:
        images = batch["images"]
        if torch.cuda.is_available():
            images = images.to(DEVICE, non_blocking=True)
            torch.cuda.synchronize()
        samples += int(images.shape[0])
        steps += 1
        if (time.time() - started) >= per_combo_seconds:
            break

    elapsed = max(1e-6, time.time() - started)
    samples_per_sec = float(samples / elapsed)
    step_time_ms = float((elapsed / max(1, steps)) * 1000.0)

    results.append({
        "batch_size": int(batch_size_value),
        "num_workers": int(workers_value),
        "samples_per_sec": samples_per_sec,
        "step_time_ms": step_time_ms,
        "steps": int(steps),
    })

    del train_loader_local
    del loaders_local

results = sorted(results, key=lambda r: (-r["samples_per_sec"], r["step_time_ms"]))
best = results[0] if results else {
    "batch_size": base_batch,
    "num_workers": base_workers,
    "samples_per_sec": 0.0,
    "step_time_ms": 0.0,
    "steps": 0,
}

print("Micro-benchmark results (top 5):")
for row in results[:5]:
    print(
        f" - batch={row['batch_size']}, workers={row['num_workers']}: "
        f"{row['samples_per_sec']:.2f} samples/s, {row['step_time_ms']:.1f} ms/step, steps={row['steps']}"
    )

print()
print(
    f"Recommended params -> batch_size={best['batch_size']}, num_workers={best['num_workers']} "
    f"({best['samples_per_sec']:.2f} samples/s)"
)

cfg["batch_size"] = int(best["batch_size"])
cfg["num_workers"] = int(best["num_workers"])
STATE["widget_config"] = cfg
STATE["benchmark_recommendation"] = {
    "best": best,
    "top_results": results[:5],
}

# Rebuild train/val/test loaders with recommended settings for next steps.
best_kwargs = {}
if int(best["num_workers"]) > 0:
    best_kwargs["prefetch_factor"] = prefetch_factor
    best_kwargs["persistent_workers"] = persistent_workers

STATE["loaders"] = create_training_loaders(
    data_dir=str(runtime_data_root),
    crop=crop_name,
    batch_size=int(best["batch_size"]),
    num_workers=int(best["num_workers"]),
    use_cache=use_cache,
    pin_memory=pin_memory,
    **best_kwargs,
)

rt("Cell 4.5: loader micro-benchmark completed")

In [ ]:
from IPython.display import clear_output, display
import inspect

rt("Cell 5: training started")

if STATE.get("adapter") is None or STATE.get("loaders") is None:
    raise RuntimeError("Run Cell 4 first.")

# Always refresh to latest widget values when Cell 2 is present.
if "gather_widget_config" in globals():
    try:
        STATE["widget_config"] = gather_widget_config()
    except Exception:
        pass
if "widget_config" not in STATE:
    raise RuntimeError("Missing widget config. Run Cell 2 and click 'Validate & Start Training'.")

adapter = STATE["adapter"]
loaders = STATE["loaders"]
num_epochs = int(STATE["widget_config"]["num_epochs"] )
val_loader = loaders.get("val")
print(f"Requested epochs: {num_epochs}")

progress_bar = widgets.IntProgress(value=0, min=0, max=max(1, len(loaders["train"])), description="Batch")
status_html = widgets.HTML(value="<b>Ready.</b>")
warning_html = widgets.HTML(value="")
epoch_summary_html = widgets.HTML(value="")
loss_out = widgets.Output()
stop_button = widgets.Button(description="Stop after current batch", button_style="warning")
display(progress_bar, status_html, warning_html, epoch_summary_html, stop_button, loss_out)

train_loss_curve = []
val_loss_curve = []
val_acc_curve = []
macro_f1_curve = []
weighted_f1_curve = []
balanced_acc_curve = []
gap_curve = []
start_time = time.time()
stop_state = {"requested": False}


def _format_advisory(event):
    message = str(event.get("advisory", "") or "").strip()
    if not message:
        return ""
    severity = str(event.get("severity", "info")).lower()
    color = {"critical": "#b00020", "warning": "#b26a00"}.get(severity, "#444")
    return f"<div style='color:{color};'><b>{severity.upper()}:</b> {message}</div>"


def _format_worst_classes(event):
    items = event.get("worst_classes", []) or []
    if not items:
        return ""
    segments = []
    for row in items:
        name = row.get("class_name", "?")
        acc = float(row.get("accuracy", 0.0))
        support = int(row.get("support", 0))
        segments.append(f"{name}: acc={acc:.3f} (n={support})")
    return "<br>".join(segments)


def _request_stop(_btn):
    stop_state["requested"] = True
    warning_html.value = "<div style='color:#b26a00;'><b>Stop requested.</b> Training will stop after the current batch.</div>"


stop_button.on_click(_request_stop)


def progress_cb(event):
    if event.get("stop_requested") is True:
        warning_html.value = "<div style='color:#b26a00;'><b>Stop acknowledged.</b> Finishing current loop safely.</div>"
        return

    if "batch" in event:
        progress_bar.max = max(1, int(event.get("total_batches", 1)))
        progress_bar.value = int(event.get("batch", 0))

        elapsed = float(event.get("elapsed_sec", time.time() - start_time))
        eta = float(event.get("eta_sec", 0.0))
        lr = float(event.get("lr", 0.0))
        grad_norm = float(event.get("grad_norm", 0.0))
        step_time_ms = float(event.get("step_time_sec", 0.0)) * 1000.0
        samples_per_sec = float(event.get("samples_per_sec", 0.0))

        status_html.value = (
            f"<b>Epoch:</b> {event.get('epoch', 0)} | "
            f"<b>Batch:</b> {event.get('batch', 0)}/{event.get('total_batches', 0)} | "
            f"<b>Loss:</b> {event.get('batch_loss', 0.0):.4f} | "
            f"<b>LR:</b> {lr:.6f} | "
            f"<b>Grad Norm:</b> {grad_norm:.3f} | "
            f"<b>Step:</b> {step_time_ms:.1f}ms | "
            f"<b>Speed:</b> {samples_per_sec:.1f} samples/s | "
            f"<b>Elapsed:</b> {elapsed:.1f}s | <b>ETA:</b> {eta:.1f}s"
        )
        warning_html.value = _format_advisory(event) or warning_html.value

    if "epoch_done" in event:
        train_loss_curve.append(float(event.get("epoch_loss", 0.0)))
        if "val_loss" in event:
            val_loss_curve.append(float(event["val_loss"]))
        if "val_accuracy" in event:
            val_acc_curve.append(float(event["val_accuracy"]))
        if "macro_f1" in event:
            macro_f1_curve.append(float(event["macro_f1"]))
        if "weighted_f1" in event:
            weighted_f1_curve.append(float(event["weighted_f1"]))
        if "balanced_accuracy" in event:
            balanced_acc_curve.append(float(event["balanced_accuracy"]))
        if "generalization_gap" in event:
            gap_curve.append(float(event["generalization_gap"]))

        with loss_out:
            clear_output(wait=True)
            plt.figure(figsize=(13, 3))

            plt.subplot(1, 3, 1)
            plt.plot(range(1, len(train_loss_curve) + 1), train_loss_curve, marker="o", label="Train Loss")
            if val_loss_curve:
                plt.plot(range(1, len(val_loss_curve) + 1), val_loss_curve, marker="s", label="Val Loss")
            plt.xlabel("Epoch")
            plt.ylabel("Loss")
            plt.title("Train/Val Loss")
            plt.grid(True, alpha=0.3)
            plt.legend()

            plt.subplot(1, 3, 2)
            if val_acc_curve:
                plt.plot(range(1, len(val_acc_curve) + 1), val_acc_curve, marker="^", label="Val Accuracy")
            if macro_f1_curve:
                plt.plot(range(1, len(macro_f1_curve) + 1), macro_f1_curve, marker="d", label="Macro F1")
            if weighted_f1_curve:
                plt.plot(range(1, len(weighted_f1_curve) + 1), weighted_f1_curve, marker="x", label="Weighted F1")
            if balanced_acc_curve:
                plt.plot(range(1, len(balanced_acc_curve) + 1), balanced_acc_curve, marker="*", label="Balanced Acc")
            plt.ylim(0.0, 1.0)
            plt.xlabel("Epoch")
            plt.ylabel("Score")
            plt.title("Validation Quality Metrics")
            plt.grid(True, alpha=0.3)
            plt.legend()

            plt.subplot(1, 3, 3)
            if gap_curve:
                plt.plot(range(1, len(gap_curve) + 1), gap_curve, marker="o", label="Val Loss - Train Loss")
            plt.axhline(0.0, color="black", linewidth=1, alpha=0.5)
            plt.xlabel("Epoch")
            plt.ylabel("Gap")
            plt.title("Generalization Gap")
            plt.grid(True, alpha=0.3)
            plt.legend()

            plt.tight_layout()
            plt.show()

        epoch_summary = (
            f"<div><b>Epoch {event.get('epoch_done')}</b>: "
            f"train_loss={event.get('epoch_loss', 0.0):.4f}"
        )
        if "val_loss" in event:
            epoch_summary += f", val_loss={event.get('val_loss', 0.0):.4f}"
        if "val_accuracy" in event:
            epoch_summary += f", val_acc={event.get('val_accuracy', 0.0):.4f}"
        if "macro_f1" in event:
            epoch_summary += f", macro_f1={event.get('macro_f1', 0.0):.4f}"
        if "weighted_f1" in event:
            epoch_summary += f", weighted_f1={event.get('weighted_f1', 0.0):.4f}"
        if "balanced_accuracy" in event:
            epoch_summary += f", balanced_acc={event.get('balanced_accuracy', 0.0):.4f}"
        if "generalization_gap" in event:
            epoch_summary += f", gap={event.get('generalization_gap', 0.0):.4f}"
        epoch_summary += "</div>"

        worst_classes_html = _format_worst_classes(event)
        if worst_classes_html:
            epoch_summary += f"<div><b>Worst classes:</b><br>{worst_classes_html}</div>"

        epoch_advisory = str(event.get("epoch_advisory", "") or "").strip()
        if epoch_advisory:
            level = str(event.get("epoch_severity", "info")).upper()
            epoch_summary += f"<div><b>{level}:</b> {epoch_advisory}</div>"

        epoch_summary_html.value = epoch_summary


train_signature = inspect.signature(adapter.train_increment)
train_kwargs = {
    "num_epochs": num_epochs,
    "progress_callback": progress_cb,
}
if "val_loader" in train_signature.parameters:
    train_kwargs["val_loader"] = val_loader
else:
    warning_html.value = warning_html.value + (
        "<div style='color:#444;'><b>Note:</b> Loaded adapter does not support "
        "val_loader telemetry yet. Pull latest repo + restart runtime for full metrics.</div>"
    )
if "should_stop" in train_signature.parameters:
    train_kwargs["should_stop"] = lambda: stop_state["requested"]
else:
    warning_html.value = warning_html.value + (
        "<div style='color:#444;'><b>Note:</b> Loaded adapter does not support "
        "graceful stop callback yet.</div>"
    )

train_result = adapter.train_increment(
    loaders["train"],
    **train_kwargs,
)

elapsed_total = time.time() - start_time
STATE["history"] = train_result.get("history", {})

print("Training complete")
print("Total time (s):", round(elapsed_total, 2))
print("Stopped early:", bool(STATE["history"].get("stopped_early", False)))
print("History:", STATE["history"])
rt("Cell 5: training completed")

In [ ]:
from IPython.display import display

rt("Cell 6: OOD calibration started")

if STATE.get("adapter") is None or STATE.get("loaders") is None:
    raise RuntimeError("Run Cell 4 first.")

adapter = STATE["adapter"]
val_loader = STATE["loaders"].get("val")

if val_loader is None or len(val_loader.dataset) == 0:
    raise RuntimeError("Validation loader is empty; cannot calibrate OOD.")

cal = adapter.calibrate_ood(val_loader)
STATE["calibration"] = cal

num_classes = cal.get("ood_calibration", {}).get("num_classes", 0)
version = cal.get("ood_calibration", {}).get("version", 0)
status_color = "green" if int(num_classes) > 0 else "red"

cal_html = widgets.HTML(
    value=f"<b style='color:{status_color}'>OOD calibration completed</b> | classes={num_classes} | version={version}"
)
display(cal_html)
rt("Cell 6: OOD calibration completed")

In [ ]:
rt("Cell 7: adapter save started")

if STATE.get("adapter") is None:
    raise RuntimeError("Run Cell 4 first.")

checkpoint_dir = ROOT / "outputs" / "colab_notebook_training"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

STATE["adapter"].save_adapter(str(checkpoint_dir))
asset_dir = checkpoint_dir / "continual_sd_lora_adapter"

print("Saved adapter directory:", asset_dir)
if not asset_dir.exists():
    raise RuntimeError(f"Expected adapter dir missing: {asset_dir}")

for p in sorted(asset_dir.rglob("*")):
    if p.is_file():
        print(" -", p.relative_to(ROOT))

rt("Cell 7: adapter save completed")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

rt("Cell 8: validation metrics started")

if STATE.get("adapter") is None or STATE.get("loaders") is None:
    raise RuntimeError("Run Cell 4 first.")

adapter = STATE["adapter"]
val_loader = STATE["loaders"].get("val")
if val_loader is None or len(val_loader.dataset) == 0:
    raise RuntimeError("Validation loader is empty.")

trainer = adapter._trainer
if trainer is None:
    raise RuntimeError("Trainer not initialized.")

trainer.adapter_model.eval()
trainer.classifier.eval()
trainer.fusion.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for batch in val_loader:
        images = batch["images"].to(trainer.device)
        labels = batch["labels"].to(trainer.device)
        logits = trainer.forward_logits(images)
        preds = torch.argmax(logits, dim=1)
        y_true.extend(labels.cpu().tolist())
        y_pred.extend(preds.cpu().tolist())

if not y_true:
    raise RuntimeError("No validation samples available.")

classes = [name for name, _ in sorted(adapter.class_to_idx.items(), key=lambda kv: kv[1])]
print(classification_report(y_true, y_pred, target_names=classes, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap="Blues")
plt.title("Validation Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(range(len(classes)), classes, rotation=45, ha="right")
plt.yticks(range(len(classes)), classes)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, int(cm[i, j]), ha="center", va="center")
plt.tight_layout()
plt.show()
rt("Cell 8: validation metrics completed")